### Implementation of multi-head attention using matrix split approach
1. This approch to calculate context vectors for each token is efficient and have less computations. 
2. Matrix computations are very less as compare to Sequential method. 
3. Each context vector for a head will then be concatenated into single context vector. 

In [1]:
import torch
import torch.nn as nn

In [86]:
# Implementation of building multi-head attention using matrix split 
class MultiHeadAttentionV2(nn.Module):
    def __init__(self, d_in, d_out, dropout, num_heads, context_length, qkv_bias= False):
        # Importing all nn.Module features (abstraction)
        super().__init__()
        # Initialization of trainable query, key, value matrices 
        self.W_q = nn.Linear(d_in, d_out, bias= qkv_bias)
        self.W_k = nn.Linear(d_in, d_out, bias= qkv_bias)
        self.W_v = nn.Linear(d_in, d_out, bias= qkv_bias)
        # Calculate head_dims 
        self.head_dims = int(d_out / num_heads)
        self.num_heads = num_heads
        # Registering triu matrix in mask 
        self.register_buffer('masked_value', torch.triu(torch.ones(context_length, context_length), diagonal= 1))
        # Dropout 
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch, tokens, d_out = x.shape
        # Initialize query, key, value matrices 
        query = self.W_q(x)
        key = self.W_k(x)
        value = self.W_v(x)

        # Transformation of query, key, value matrix
        # from (batch, tokens, d_out) ----> (batch, tokens, num_head, head_dims)
        query = query.view(batch, tokens, self.num_heads, self.head_dims)
        key = key.view(batch, tokens, self.num_heads, self.head_dims)
        value = value.view(batch, tokens, self.num_heads, self.head_dims)

        # Regrouping/ odering matrix according to number of head wise 
        # (batch, tokens, num_heads, head_dims) --> (batch, num_heads, token, head_dims)
        query = query.transpose(1, 2)
        key = key.transpose(1, 2)
        value = value.transpose(1, 2)
        
        # Calculating attention scores now 
        attention_scores = query @ key.transpose(2, 3) # Transposing (token, head_dims) -> (head_dims, token)

        # Calculating attention matrix 
        # Upper Triangular Matrix masking + Scaling + softmax 
        attention_scores = attention_scores.masked_fill_(
            self.masked_value.bool(), -torch.inf
        )
        # Appling scaling + softmax 
        attention_matrix = torch.softmax(attention_scores / key.shape[-1]**0.5, dim= -1)

        # Applying Dropout: for generalization + help in resolving vanishing gradient problem 
        self.dropout(attention_matrix)

        # Calculating context vectors 
        context_vecs = (attention_matrix @ value).transpose(1, 2) # Reverting to (batch, num_token, num_head, head_dims)

        # Re-arranging the context vector back to original dims 
        # original Dims: - (batch, tokens, d_out)
        context_vecs = context_vecs.contiguous().view(batch, tokens, d_out)
        return context_vecs

In [2]:
# Input token values
inputs = torch.tensor([
    [0.43, 0.15, 0.89, 0.76, 0.80, 0.64], # The
    [0.55, 0.87, 0.66, 0.54, 0.48, 0.71], # Cat
    [0.57, 0.85, 0.64, 0.40, 0.92, 0.61], # Sleeps
])

In [87]:
# Calculating attention scores 
torch.manual_seed(123)
batch = torch.stack((inputs, inputs), dim= 0) # Stack tensors on top of each other.
d_in = 6 # Number of feature dimensions used in each token 
d_out = 6 # Same as input dims 
mha = MultiHeadAttentionV2(d_in, d_out, 0.5, 2, 3)
print(f"Context.Vectors: \n{mha.forward(batch)}")

Context.Vectors: 
tensor([[[-0.1373,  0.2521,  0.1879,  0.1681,  0.4830,  0.0787],
         [-0.3220,  0.3126,  0.0860,  0.2390,  0.5137,  0.0330],
         [-0.3521,  0.3426,  0.0631,  0.2234,  0.5843,  0.0139]],

        [[-0.1373,  0.2521,  0.1879,  0.1681,  0.4830,  0.0787],
         [-0.3220,  0.3126,  0.0860,  0.2390,  0.5137,  0.0330],
         [-0.3521,  0.3426,  0.0631,  0.2234,  0.5843,  0.0139]]],
       grad_fn=<ViewBackward0>)


In [88]:
# Example: - Building intuition with examples, using first-principle thinking. 
batch = torch.stack((inputs, inputs), dim= 0) # Stack tensors on top of each other. 
d_in = 6 # Number of feature dimensions used in each token 
d_out = 6 # Same as input dims 

W_q = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad= False)
W_k = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad= False)
W_v = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad= False)

# From these initial matrix calculating query, key, value matrix 
query = batch @ W_q
key = batch @ W_k
values = batch @ W_v

# Transforming each matrix dims from (batches, tokens, d_out) = (batchs, tokens, num_heads, head_dims)
num_heads = 2 # Each head has it's context vector 
head_dims = int(d_in / num_heads)

# Transformation 
batches, tokens, d_out = batch.shape
query = query.view(batches, tokens, num_heads, head_dims)
key = key.view(batches, tokens, num_heads, head_dims)
values = values.view(batches, tokens, num_heads, head_dims)

# As matrices are grouped in number of token-wise
# Now, we need to group matrices according to number of head wise
# (batchs, tokens, num_head, head_dims) ---> (batchs, num_head, tokens, head_dims)
query = query.transpose(1, 2) 
key = key.transpose(1, 2)
values = values.transpose(1, 2)

# Getting the attention score = query @ key.Transpose
atttention_score = query @ key.transpose(2, 3)
print(f'attention.score: \n{atttention_score}')

attention.score: 
tensor([[[[-2.9237, -1.8538, -2.2779],
          [-0.0367,  0.2951,  0.0502],
          [ 0.3819,  0.9354,  0.3260]],

         [[ 5.6841,  5.4599,  5.8157],
          [ 5.7529,  6.1821,  7.0912],
          [ 4.3650,  4.1722,  4.5651]]],


        [[[-2.9237, -1.8538, -2.2779],
          [-0.0367,  0.2951,  0.0502],
          [ 0.3819,  0.9354,  0.3260]],

         [[ 5.6841,  5.4599,  5.8157],
          [ 5.7529,  6.1821,  7.0912],
          [ 4.3650,  4.1722,  4.5651]]]])


In [30]:
# masking with -inf upper triangular matrix + scaling + applying softmax function 
context_length = 3 # Number of tokens considering per batch 
mask = torch.triu(torch.ones(context_length, context_length), diagonal= 1)
# In-place masking to replace upper triangular values to -inf 
atttention_score = atttention_score.masked_fill_(mask.bool(), -torch.inf)
# Calculating softmax and normalizing in - place of column 
atttention_score = torch.softmax(atttention_score / key.shape[-1]**0.5, dim= -1) # Normalizing along column
print(f"att_scores.values: \n{atttention_score}")

att_scores.values: 
tensor([[[[1.0000, 0.0000, 0.0000],
          [0.5372, 0.4628, 0.0000],
          [0.3591, 0.2885, 0.3524]],

         [[1.0000, 0.0000, 0.0000],
          [0.4628, 0.5372, 0.0000],
          [0.3064, 0.3813, 0.3123]]],


        [[[1.0000, 0.0000, 0.0000],
          [0.5372, 0.4628, 0.0000],
          [0.3591, 0.2885, 0.3524]],

         [[1.0000, 0.0000, 0.0000],
          [0.4628, 0.5372, 0.0000],
          [0.3064, 0.3813, 0.3123]]]])


#### Below is illustration of the above class module — built from first principles

---

#### Why Multi-Head Attention?

Single-head attention computes **one** set of attention weights — the model can only focus on one relationship pattern at a time (e.g., syntactic structure). Multi-head attention splits the representation space so that **each head can independently learn different relationships** (e.g., one head captures positional proximity, another captures semantic similarity).

---

#### Step-by-step Walkthrough

We have **3 tokens** (`The`, `Cat`, `Sleeps`), each represented by a **6-dimensional** embedding vector.

---

**Step 1 — Batching**

```
batch = torch.stack((inputs, inputs), dim=0)   # shape: (2, 3, 6)
```

We simulate a batch of 2 identical sequences. Each sequence has 3 tokens × 6 features.

```
batch.shape = (batch_size=2, tokens=3, d_in=6)
```

---

**Step 2 — Project into Query, Key, Value spaces**

```
query  = batch @ W_q    # (2, 3, 6) @ (6, 6) → (2, 3, 6)
key    = batch @ W_k    # same
values = batch @ W_v    # same
```

Each token gets transformed into three different representations:
| Matrix | Purpose |
|--------|---------|
| **Query** | "What am I looking for?" |
| **Key** | "What do I contain?" |
| **Value** | "What information do I carry?" |

All three still have shape `(batch=2, tokens=3, d_out=6)`.

---

**Step 3 — Reshape: split `d_out` into `num_heads × head_dims`**

With `num_heads = 2` and `head_dims = 6 / 2 = 3`:

```
query = query.view(batch, tokens, num_heads, head_dims)
# (2, 3, 6) → (2, 3, 2, 3)
```

Visually, each token's 6-dim vector is **split in half** — the first 3 dims go to Head 0, the last 3 dims go to Head 1:

```
Token "The" = [f0, f1, f2, f3, f4, f5]
                ├── Head 0: [f0, f1, f2]
                └── Head 1: [f3, f4, f5]
```

---

**Step 4 — Transpose: regroup from token-wise to head-wise**

```
query = query.transpose(1, 2)
# (2, 3, 2, 3) → (2, 2, 3, 3)
#  B  T  H  D     B  H  T  D
```

**Why?** Before transpose, data is organized as: *for each token, here are its heads*. After transpose: *for each head, here are all tokens*. This lets us compute attention **independently per head** using a single batched matrix multiply.

```
Before transpose:               After transpose:
[batch, tokens, heads, dims]    [batch, heads, tokens, dims]

  Token 0 → [Head0 | Head1]      Head 0 → [Token0, Token1, Token2]
  Token 1 → [Head0 | Head1]      Head 1 → [Token0, Token1, Token2]
  Token 2 → [Head0 | Head1]
```

---

**Step 5 — Compute attention scores**

```
attention_score = query @ key.transpose(2, 3)
# (2, 2, 3, 3) @ (2, 2, 3, 3)ᵀ → (2, 2, 3, 3)
#  B  H  T  D     B  H  D  T      B  H  T  T
```

For each head, we get a `(tokens × tokens)` matrix — each entry `[i, j]` tells us **how much token i should attend to token j** within that head's subspace.

```
Head 0 attention scores:        Head 1 attention scores:
      The  Cat  Sleeps               The  Cat  Sleeps
The  [ .    .    .   ]          The  [ .    .    .   ]
Cat  [ .    .    .   ]          Cat  [ .    .    .   ]
Slp  [ .    .    .   ]          Slp  [ .    .    .   ]

↑ Each head learns DIFFERENT       ↑ Different patterns
  attention patterns                  in a different subspace
```

---

#### What happens next (in the class module)

| Step | Operation | Purpose |
|------|-----------|---------|
| **Masking** | Upper triangular → `-inf` | Prevents tokens from attending to future tokens (causal) |
| **Scaling** | Divide by `√head_dims` | Prevents softmax saturation for large dimensions |
| **Softmax** | Along last dim | Converts raw scores to probabilities (rows sum to 1) |
| **Dropout** | Random zeroing | Regularization to prevent overfitting |
| **Context vectors** | `attention_matrix @ values` | Weighted sum of value vectors |
| **Concat heads** | `.transpose(1,2).contiguous().view(...)` | Merge all heads back → `(batch, tokens, d_out)` |

---

#### The Big Picture

```
Input (2, 3, 6)
    │
    ├──→ W_q ──→ Q (2,3,6) ──→ reshape ──→ (2,3,2,3) ──→ transpose ──→ (2,2,3,3)
    ├──→ W_k ──→ K (2,3,6) ──→ reshape ──→ (2,3,2,3) ──→ transpose ──→ (2,2,3,3)
    └──→ W_v ──→ V (2,3,6) ──→ reshape ──→ (2,3,2,3) ──→ transpose ──→ (2,2,3,3)
                                                                │
                                              Q @ Kᵀ = attention scores (2,2,3,3)
                                                                │
                                              mask + scale + softmax + dropout
                                                                │
                                              scores @ V = context vectors (2,2,3,3)
                                                                │
                                              transpose + reshape → (2,3,6)
                                                                │
                                                        Output (2, 3, 6)
```

Each head operates on a **3-dimensional slice** of the full 6-dimensional space, learns its own attention pattern, and the results are concatenated back to the original dimensionality.